In [228]:
import pandas as pd
import numpy as np

In [229]:
df_raw=pd.read_csv(r'D:\Programming\Python\Data Analysis\Startup_Funding_Analysis\data\startup_funding.csv')
df=df_raw.copy()

In [230]:
print("shape: \n",df.shape)
print('info: \n',df.info())
print('describe: \n',df.describe())
print('dtypes: \n',df.dtypes)

shape: 
 (3044, 10)
<class 'pandas.DataFrame'>
RangeIndex: 3044 entries, 0 to 3043
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Sr No              3044 non-null   int64
 1   Date dd/mm/yyyy    3044 non-null   str  
 2   Startup Name       3044 non-null   str  
 3   Industry Vertical  2873 non-null   str  
 4   SubVertical        2108 non-null   str  
 5   City  Location     2864 non-null   str  
 6   Investors Name     3020 non-null   str  
 7   InvestmentnType    3040 non-null   str  
 8   Amount in USD      2084 non-null   str  
 9   Remarks            419 non-null    str  
dtypes: int64(1), str(9)
memory usage: 237.9 KB
info: 
 None
describe: 
              Sr No
count  3044.000000
mean   1522.500000
std     878.871435
min       1.000000
25%     761.750000
50%    1522.500000
75%    2283.250000
max    3044.000000
dtypes: 
 Sr No                int64
Date dd/mm/yyyy        str
Startup Name           s

In [231]:
# converting amount in usd column to numeric dtype
# we need to replace commas in there
df['Amount in USD']=df['Amount in USD'].str.replace(',','',regex=False)
# now convert it to numeric dtype
df['Amount in USD']=pd.to_numeric(df['Amount in USD'],errors='coerce')


In [232]:
print('datatypes: \n',df.dtypes)
print('describe: \n',df.describe(),'\n\n\n')
print('amount in usd.isna.sum:',df['Amount in USD'].isna().sum())

datatypes: 
 Sr No                  int64
Date dd/mm/yyyy          str
Startup Name             str
Industry Vertical        str
SubVertical              str
City  Location           str
Investors Name           str
InvestmentnType          str
Amount in USD        float64
Remarks                  str
dtype: object
describe: 
              Sr No  Amount in USD
count  3044.000000   2.065000e+03
mean   1522.500000   1.842990e+07
std     878.871435   1.213734e+08
min       1.000000   1.600000e+04
25%     761.750000   4.700000e+05
50%    1522.500000   1.700000e+06
75%    2283.250000   8.000000e+06
max    3044.000000   3.900000e+09 



amount in usd.isna.sum: 979


In [233]:
#cleaning/converting thr date column into proper format
print(df['Date dd/mm/yyyy'].dtype)
# convert the date column to yyyy/mm/dd i.e mysql accetable format 
# and also change the datatype of column from string to datetime
df['Date dd/mm/yyyy']=pd.to_datetime(df['Date dd/mm/yyyy'],dayfirst=True,errors='coerce')

str


In [234]:
print('nulls in date:',df['Date dd/mm/yyyy'].isna().sum())
print(df.dtypes)

nulls in date: 8
Sr No                         int64
Date dd/mm/yyyy      datetime64[us]
Startup Name                    str
Industry Vertical               str
SubVertical                     str
City  Location                  str
Investors Name                  str
InvestmentnType                 str
Amount in USD               float64
Remarks                         str
dtype: object


In [235]:
bad_rows = df["Date dd/mm/yyyy"].isna()

print(df_raw.loc[bad_rows, ["Sr No", "Date dd/mm/yyyy", "Startup Name"]])

      Sr No      Date dd/mm/yyyy                  Startup Name
192     193            05/072018                      Loan Tap
2571   2572            01/07/015                     HandyHome
2606   2607  \\xc2\\xa010/7/2015  \\xc2\\xa0Infinity Assurance
2775   2776           12/05.2015                      Mobiefit
2776   2777           12/05.2015                      Plancess
2831   2832           13/04.2015                    The Porter
3011   3012           15/01.2015                     Wishberry
3029   3030          22/01//2015                  Corporate360


In [236]:
# Date Corrections
# 05/072018  -> 05/07/2018
# 01/07/015  -> 01/07/2015
# Remove leading '\xc2\xa0' -> 10/07/2015
# 12/05.2015 -> 12/05/2015
# 13/04.2015 -> 13/04/2015
# 15/01.2015 -> 15/01/2015
# 22/01//2015 -> 22/01/2015
# 
# most of the data in these dates columns is salvagable so we will just clean them up

In [237]:
# correction code for those bad dates
bad_rows = df["Date dd/mm/yyyy"].isna()
bad_dates = df_raw.loc[bad_rows, "Date dd/mm/yyyy"].copy()



In [238]:
bad_dates = bad_dates.str.replace(".", "/", regex=False)
bad_dates = bad_dates.str.replace("//", "/", regex=False)
bad_dates = bad_dates.str.replace(r"\\\\xc2\\\\xa010/7/2015", "10/07/2015", regex=True)

bad_dates = bad_dates.replace({
    "05/072018": "05/07/2018",
    "01/07/015": "01/07/2015"
})

In [239]:
corrected_dates = pd.to_datetime(
    bad_dates,
    dayfirst=True,
    errors="coerce"
)

In [240]:
df.loc[bad_rows, "Date dd/mm/yyyy"] = corrected_dates

In [241]:
#verify
print("Remaining invalid dates:", df["Date dd/mm/yyyy"].isna().sum())

Remaining invalid dates: 0


In [242]:
# cleaning of column startup name for any unusual encoding issues
print('unique startup names\n',df['Startup Name'].nunique())
print('first 30 startup names\n',df["Startup Name"].sort_values().head(30))
print('last 30 startup names\n',df["Startup Name"].sort_values().tail(30))
print('str contains\n',df[df["Startup Name"].str.contains("â", na=False)])


unique startup names
 2459
first 30 startup names
 67                "BYJU\\'S"
2992                   #Fame
1242               121Policy
201                19th mile
1007                  1Crowd
311                      1mg
1737                     1mg
1793                     1mg
716                      1mg
2844    1mg (Healthkartplus)
2964               20Dresses
2595               33Coupons
2012                 360Ride
1307                 3Dexter
154                   3HCare
595                   3HCare
38                   3rdFlix
1009                  48East
788                    4tigo
202                5th Vital
2229                 6Degree
1581                 6Degree
39                       75F
613            91SpringBoard
1436           91SpringBoard
1104                 99Games
1106               99PerHour
462                  9Stacks
130          A&R Bon Vivants
735               ABI Health
Name: Startup Name, dtype: str
last 30 startup names
 217                iqlect

In [243]:
print((df["Startup Name"] != df["Startup Name"].str.strip()).sum())


0


In [244]:
print(df["Startup Name"].str.contains(r"\s{2,}", regex=True, na=False).sum())

0


In [245]:
print(df[df["Startup Name"].str.contains("BYJU", na=False)])

      Sr No Date dd/mm/yyyy          Startup Name  Industry Vertical  \
0         1      2020-01-09                BYJU’S             E-Tech   
67       68      2019-07-10            "BYJU\\'S"             EdTech   
1164   1165      2016-12-20  BYJU\\xe2\\x80\\x99s  Consumer Internet   

                   SubVertical City  Location  \
0                   E-learning      Bengaluru   
67                   Education      Bengaluru   
1164  Online Learning platform      Bangalore   

                           Investors Name       InvestmentnType  \
0                 Tiger Global Management  Private Equity Round   
67             Qatar Investment Authority  Private Equity Round   
1164  International Financial Corporation        Private Equity   

      Amount in USD Remarks  
0       200000000.0     NaN  
67      150000000.0     NaN  
1164     15000000.0     NaN  


In [246]:
for name in df[df["Startup Name"].str.contains("BYJU", na=False)]["Startup Name"]:
    print(repr(name))

'BYJU’S'
'"BYJU\\\\\'S"'
'BYJU\\\\xe2\\\\x80\\\\x99s'


In [247]:
# changing the name of company to proper name
df["Startup Name"] = df["Startup Name"].replace({
    "BYJU’S": "BYJU'S",
    "\"BYJU\\\\'S\"": "BYJU'S",
    "BYJU\\\\xe2\\\\x80\\\\x99s": "BYJU'S"
})

In [248]:
print(df[df["Startup Name"].str.contains("BYJU", na=False)])

      Sr No Date dd/mm/yyyy Startup Name  Industry Vertical  \
0         1      2020-01-09       BYJU'S             E-Tech   
67       68      2019-07-10       BYJU'S             EdTech   
1164   1165      2016-12-20       BYJU'S  Consumer Internet   

                   SubVertical City  Location  \
0                   E-learning      Bengaluru   
67                   Education      Bengaluru   
1164  Online Learning platform      Bangalore   

                           Investors Name       InvestmentnType  \
0                 Tiger Global Management  Private Equity Round   
67             Qatar Investment Authority  Private Equity Round   
1164  International Financial Corporation        Private Equity   

      Amount in USD Remarks  
0       200000000.0     NaN  
67      150000000.0     NaN  
1164     15000000.0     NaN  


In [249]:
df["Startup Name"].nunique()

2457

In [250]:
# now cleaning of city location
df["City  Location"].nunique()

112

In [251]:
df["City  Location"].isna().sum()

np.int64(180)

In [252]:
print(df["City  Location"].value_counts().to_dict())

{'Bangalore': 700, 'Mumbai': 567, 'New Delhi': 421, 'Gurgaon': 287, 'Bengaluru': 141, 'Pune': 105, 'Hyderabad': 99, 'Chennai': 97, 'Noida': 92, 'Gurugram': 50, 'Ahmedabad': 38, 'Delhi': 34, 'Jaipur': 30, 'Kolkata': 21, 'Indore': 13, 'Chandigarh': 11, 'Goa': 10, 'Vadodara': 10, 'Singapore': 8, 'Coimbatore': 5, 'Kanpur': 4, 'Pune / US': 4, '\\\\xc2\\\\xa0Gurgaon': 4, 'Faridabad': 3, 'Bhopal': 3, 'Nagpur': 3, '\\\\xc2\\\\xa0New Delhi': 3, 'San Francisco': 2, 'Kormangala': 2, 'Mumbai/Bengaluru': 2, 'India/US': 2, 'Ahemadabad': 2, 'Udaipur': 2, 'Surat': 2, 'Trivandrum': 2, 'Gwalior': 2, 'Udupi': 2, 'Kochi': 2, 'Agra': 2, 'Bangalore/ Bangkok': 2, 'Siliguri': 2, 'Bangalore / SFO': 2, 'New Delhi / US': 2, 'San Jose,': 1, 'Amritsar': 1, 'Tulangan': 1, 'Burnsville': 1, 'Menlo Park': 1, 'Palo Alto': 1, 'Santa Monica': 1, 'Taramani': 1, 'Andheri': 1, 'Chembur': 1, 'Nairobi': 1, 'Haryana': 1, 'New York': 1, 'Karnataka': 1, 'Bengaluru and Gurugram': 1, 'India/Singapore': 1, 'New York, Bengaluru': 1,

In [253]:
# Bangalore      -> Bengaluru
# Gurgaon        -> Gurugram
# Ahemadabad     -> Ahmedabad
# Ahemdabad      -> Ahmedabad
# Kolkatta       -> Kolkata
# Bhubneswar     -> Bhubaneswar
# Nw Delhi       -> New Delhi
# \\\\xc2\\\\xa0Bangalore -> Bangalore
# \\\\xc2\\\\xa0Mumbai    -> Mumbai
# \\xc2\\xa0New Delhi -> New Delhi
# \\\\xc2\\\\xa0Noida     -> Noida
# \\xc2\\xa0Gurgaon   -> Gurgaon


In [254]:
# dict_citylocation=df["City  Location"].value_counts().to_dict()
# print(dict_citylocation)

city_replacements = {
    # Official city names
    "Bangalore": "Bengaluru",
    "Gurgaon": "Gurugram",

    # Typographical errors
    "Ahemadabad": "Ahmedabad",
    "Ahemdabad": "Ahmedabad",
    "Kolkatta": "Kolkata",
    "Bhubneswar": "Bhubaneswar",
    "Nw Delhi": "New Delhi",

    # Encoding issues
    "\\\\xc2\\\\xa0Bangalore": "Bengaluru",
    "\\\\xc2\\\\xa0Mumbai": "Mumbai",
    "\\\\xc2\\\\xa0New Delhi": "New Delhi",
    "\\\\xc2\\\\xa0Noida": "Noida",
    "\\\\xc2\\\\xa0Gurgaon": "Gurugram"
}

df["City  Location"] = df["City  Location"].replace(city_replacements)

In [255]:
print(df["City  Location"].value_counts().to_dict())

{'Bengaluru': 842, 'Mumbai': 568, 'New Delhi': 425, 'Gurugram': 341, 'Pune': 105, 'Hyderabad': 99, 'Chennai': 97, 'Noida': 93, 'Ahmedabad': 41, 'Delhi': 34, 'Jaipur': 30, 'Kolkata': 22, 'Indore': 13, 'Chandigarh': 11, 'Goa': 10, 'Vadodara': 10, 'Singapore': 8, 'Coimbatore': 5, 'Kanpur': 4, 'Pune / US': 4, 'Faridabad': 3, 'Bhopal': 3, 'Nagpur': 3, 'San Francisco': 2, 'Kormangala': 2, 'Mumbai/Bengaluru': 2, 'India/US': 2, 'Bhubaneswar': 2, 'Udaipur': 2, 'Surat': 2, 'Trivandrum': 2, 'Gwalior': 2, 'Udupi': 2, 'Kochi': 2, 'Agra': 2, 'Bangalore/ Bangkok': 2, 'Siliguri': 2, 'Bangalore / SFO': 2, 'New Delhi / US': 2, 'San Jose,': 1, 'Amritsar': 1, 'Tulangan': 1, 'Burnsville': 1, 'Menlo Park': 1, 'Palo Alto': 1, 'Santa Monica': 1, 'Taramani': 1, 'Andheri': 1, 'Chembur': 1, 'Nairobi': 1, 'Haryana': 1, 'New York': 1, 'Karnataka': 1, 'Bengaluru and Gurugram': 1, 'India/Singapore': 1, 'New York, Bengaluru': 1, 'California': 1, 'India': 1, 'Rourkela': 1, 'Srinagar': 1, 'Delhi & Cambridge': 1, 'Uttar

In [256]:
# industry vertical

print('unique:',df["Industry Vertical"].nunique())
print('nulls:',df["Industry Vertical"].isna().sum())
print(df["Industry Vertical"].value_counts(dropna=False).to_dict())

unique: 821
nulls: 171
{'Consumer Internet': 941, 'Technology': 478, 'eCommerce': 186, nan: 171, 'Healthcare': 70, 'Finance': 62, 'ECommerce': 61, 'Logistics': 32, 'E-Commerce': 29, 'Education': 24, 'Food & Beverage': 23, 'Ed-Tech': 14, 'E-commerce': 12, 'FinTech': 9, 'Ecommerce': 8, 'IT': 8, 'Food and Beverage': 6, 'Real Estate': 6, 'Fin-Tech': 6, 'Others': 6, 'Health and Wellness': 5, 'Logistics Tech': 5, 'Online Education Platform': 5, 'Online Food Delivery': 5, 'Transportation': 4, 'Transport': 4, 'SaaS': 3, 'Information Technology': 3, 'EdTech': 3, 'Services': 3, 'Automobile': 3, 'Social Media': 3, 'Food': 3, 'Food & Beverages': 3, 'Food and Beverages': 3, 'ecommerce': 3, 'FMCG': 3, 'Food Delivery Platform': 3, 'Big Data & Analytics platform': 3, 'Hyperlocal Handyman Services': 3, 'Hospitality': 2, 'Gaming': 2, 'Software': 2, 'Last Mile Transportation': 2, 'B2B': 2, 'Consumer Goods': 2, 'Tech': 2, 'Health Care': 2, 'Waste Management Service': 2, 'Agriculture': 2, 'Digital Media': 

In [257]:
industry_replacements = {

    
    # E-commerce
    
    "eCommerce": "E-commerce",
    "ECommerce": "E-commerce",
    "Ecommerce": "E-commerce",
    "ecommerce": "E-commerce",
    "E-Commerce": "E-commerce",

  
    # Education
  
    "Ed-Tech": "EdTech",
    "Ed-tech": "EdTech",
    "Edtech": "EdTech",
    "E-Tech": "EdTech",

    
    # Finance

    "Fin-Tech": "FinTech",
    "Fintech": "FinTech",
    "Financial Tech": "FinTech",
    "Fiinance": "Finance",

    # Technology

    "IT": "Technology",
    "Information Technology": "Technology",
    "Tech": "Technology",

    # Healthcare

    "Health Care": "Healthcare",
    "healthcare": "Healthcare",
    "Health and wellness": "Health and Wellness",

    # Food & Beverage

    "Food and Beverage": "Food & Beverage",
    "Food & Beverages": "Food & Beverage",
    "Food and Beverages": "Food & Beverage"
}
df["Industry Vertical"] = df["Industry Vertical"].replace(industry_replacements)

In [258]:
print(df["Industry Vertical"].value_counts(dropna=False).head(25))

Industry Vertical
Consumer Internet            941
Technology                   491
E-commerce                   299
NaN                          171
Healthcare                    73
Finance                       63
Food & Beverage               35
Logistics                     32
Education                     24
EdTech                        21
FinTech                       18
Health and Wellness            6
Real Estate                    6
Others                         6
Logistics Tech                 5
Online Education Platform      5
Online Food Delivery           5
Transportation                 4
Transport                      4
SaaS                           3
Services                       3
Automobile                     3
Social Media                   3
Food                           3
FMCG                           3
Name: count, dtype: int64


In [259]:
# Investment column
print("Unique:", df["InvestmentnType"].nunique())

print("Nulls:", df["InvestmentnType"].isna().sum())

print(df["InvestmentnType"].value_counts(dropna=False).to_dict())

Unique: 55
Nulls: 4
{'Private Equity': 1356, 'Seed Funding': 1355, 'Seed/ Angel Funding': 60, 'Seed / Angel Funding': 47, 'Seed\\\\nFunding': 30, 'Debt Funding': 25, 'Series A': 24, 'Seed/Angel Funding': 23, 'Series B': 20, 'Series C': 14, 'Series D': 12, 'Angel / Seed Funding': 8, 'Seed Round': 7, 'Private Equity Round': 4, 'Seed': 4, nan: 4, 'Pre-Series A': 4, 'Seed / Angle Funding': 3, 'Series F': 2, 'Series E': 2, 'Corporate Round': 2, 'Venture Round': 2, 'pre-Series A': 2, 'Equity': 2, 'Pre-series A': 1, 'Series G': 1, 'Series H': 1, 'Venture': 1, 'Funding Round': 1, 'Maiden Round': 1, 'pre-series A': 1, 'Seed Funding Round': 1, 'Single Venture': 1, 'Angel': 1, 'Series J': 1, 'Angel Round': 1, 'Venture - Series Unknown': 1, 'Bridge Round': 1, 'Debt and Preference capital': 1, 'Inhouse Funding': 1, 'Debt': 1, 'Pre Series A': 1, 'Debt-Funding': 1, 'Mezzanine': 1, 'Series B (Extension)': 1, 'Equity Based Funding': 1, 'Private Funding': 1, 'Seed funding': 1, 'Private': 1, 'Structured 

In [260]:
# cleaning bad investment names
investment_replacements = {

    # Seed Funding
    "Seed/ Angel Funding": "Seed Funding",
    "Seed / Angel Funding": "Seed Funding",
    "Seed/Angel Funding": "Seed Funding",
    "Seed\\nFunding": "Seed Funding",
    "Seed Round": "Seed Funding",
    "Seed Funding Round": "Seed Funding",
    "Seed funding": "Seed Funding",
    "Seed": "Seed Funding",

    # Typo
    "Seed / Angle Funding": "Seed Funding",

    # Angel Funding
    "Angel": "Angel Funding",
    "Angel Round": "Angel Funding",

    # I would keep this separate because it explicitly says both
    # Angel / Seed Funding

    # Private Equity
    "Private Equity Round": "Private Equity",
    "PrivateEquity": "Private Equity",
    "Private\\nEquity": "Private Equity",
    "Private": "Private Equity",
    "Private\\\\nEquity": "Private Equity",

    # Debt
    "Debt": "Debt Funding",
    "Debt-Funding": "Debt Funding",

    # Pre-Series A
    "Pre-Series A": "Pre-Series A",
    "pre-Series A": "Pre-Series A",
    "pre-series A": "Pre-Series A",
    "Pre Series A": "Pre-Series A",
    "Pre-Series A": "Pre-Series A",
    "Pre-series A": "Pre-Series A",
    # Equity
    "Equity Based Funding": "Equity",

    # Crowd Funding
    "Crowd funding": "Crowd Funding"
}
df["InvestmentnType"] = df["InvestmentnType"].replace(investment_replacements)

In [261]:
print(df["InvestmentnType"].value_counts(dropna=False).to_dict())

{'Seed Funding': 1501, 'Private Equity': 1363, 'Seed\\\\nFunding': 30, 'Debt Funding': 27, 'Series A': 24, 'Series B': 20, 'Series C': 14, 'Series D': 12, 'Pre-Series A': 9, 'Angel / Seed Funding': 8, nan: 4, 'Angel Funding': 3, 'Equity': 3, 'Series F': 2, 'Series E': 2, 'Corporate Round': 2, 'Venture Round': 2, 'Crowd Funding': 2, 'Series G': 1, 'Series H': 1, 'Venture': 1, 'Funding Round': 1, 'Maiden Round': 1, 'Single Venture': 1, 'Series J': 1, 'Venture - Series Unknown': 1, 'Bridge Round': 1, 'Debt and Preference capital': 1, 'Inhouse Funding': 1, 'Mezzanine': 1, 'Series B (Extension)': 1, 'Private Funding': 1, 'Structured Debt': 1, 'Term Loan': 1}


In [262]:
# investors name
print("Unique:", df["Investors Name"].nunique())
print("Nulls:", df["Investors Name"].isna().sum())
print(df["Investors Name"].value_counts(dropna=False).to_dict())

Unique: 2412
Nulls: 24
{'Undisclosed Investors': 39, 'Undisclosed investors': 30, 'Ratan Tata': 25, nan: 24, 'Indian Angel Network': 23, 'Kalaari Capital': 16, 'Sequoia Capital': 15, 'Group of Angel Investors': 15, 'Accel Partners': 12, 'Undisclosed Investor': 12, 'Undisclosed': 11, 'Venture Catalysts': 11, 'Brand Capital': 11, 'undisclosed investors': 11, 'SAIF Partners': 10, 'RoundGlass Partners': 10, 'Nexus Venture Partners': 9, 'Info Edge (India) Ltd': 9, 'Undisclosed investor': 9, 'Blume Ventures': 8, 'Tiger Global': 8, 'Trifecta Capital': 8, 'Unitus Seed Fund': 8, 'Tiger Global Management': 7, 'Matrix Partners': 7, 'YouWeCan Ventures': 7, 'Y Combinator': 6, 'The Chennai Angels': 6, 'Bessemer Venture Partners': 6, 'Exfinity Venture Partners': 5, 'Sixth Sense Ventures': 5, 'InnoVen Capital': 5, 'Flipkart': 5, 'LetsVenture': 5, 'Omidyar Network': 5, 'IDG Ventures': 5, 'Paytm': 5, 'Axilor Ventures': 5, 'Kae Capital': 5, 'SRI Capital': 5, 'India Educational Investment Fund': 5, 'India

In [263]:
print(df[df["Investors Name"].str.contains(r"\\\\", regex=True, na=False)])

      Sr No Date dd/mm/yyyy          Startup Name  \
112     113      2019-02-01                FleetX   
116     117      2019-01-03              CarDekho   
127     128      2018-11-03  Veritas Finance Ltd.   
141     142      2018-11-24           Engineer.ai   
152     153      2018-09-03                 Udaan   
...     ...             ...                   ...   
2860   2861      2015-04-27            LogicRoots   
2891   2892      2015-03-13           DiabetOmics   
2905   2906      2015-03-19         Graphic India   
2932   2933      2015-03-31             Appsdaily   
2978   2979      2015-02-24  Caravan Craft Retail   

                  Industry Vertical             SubVertical City  Location  \
112                              AI               Logistics       Gurugram   
116                      Automobile      Online Marketplace         Jaipur   
127                            NBFC            MSME Finance        Chennai   
141                        Software             AI 

In [264]:
# checking whitespace
print(
    (df["Investors Name"] !=
     df["Investors Name"].str.strip()).sum()
)

24


In [265]:
# collapsing undisclosed investment to single entity
investor_replacements = {
    "Undisclosed Investors": "Undisclosed Investors",
    "Undisclosed investors": "Undisclosed Investors",
    "undisclosed investors": "Undisclosed Investors",
    "Undisclosed Investor": "Undisclosed Investors",
    "Undisclosed investor": "Undisclosed Investors",
    "undisclosed investor": "Undisclosed Investors",
    "Undisclosed": "Undisclosed Investors"
}

df["Investors Name"] = df["Investors Name"].replace(investor_replacements)

In [266]:
df["Investors Name"] = (
    df["Investors Name"]
      .str.replace(r"\\\\xc2\\\\xa0", " ", regex=True)
      .str.replace(r"\\\\xe2\\\\x80\\\\x99", "'", regex=True)
      .str.replace(r"\\\\n", " ", regex=True)
)

In [267]:
#removing white spaces
df["Investors Name"] = df["Investors Name"].str.strip()
df["Investors Name"] = df["Investors Name"].str.replace(
    r"\s+",
    " ",
    regex=True
)

In [268]:
print(df["Investors Name"].value_counts(dropna=False).to_dict())

{'Undisclosed Investors': 115, 'Ratan Tata': 25, nan: 24, 'Indian Angel Network': 24, 'Kalaari Capital': 16, 'Sequoia Capital': 15, 'Group of Angel Investors': 15, 'Accel Partners': 12, 'Venture Catalysts': 11, 'Brand Capital': 11, 'SAIF Partners': 10, 'RoundGlass Partners': 10, 'Nexus Venture Partners': 9, 'Tiger Global': 9, 'Info Edge (India) Ltd': 9, 'Blume Ventures': 8, 'Trifecta Capital': 8, 'Unitus Seed Fund': 8, 'Tiger Global Management': 7, 'Matrix Partners': 7, 'YouWeCan Ventures': 7, 'Y Combinator': 6, 'The Chennai Angels': 6, 'Bessemer Venture Partners': 6, 'Exfinity Venture Partners': 5, 'Sixth Sense Ventures': 5, 'InnoVen Capital': 5, 'Flipkart': 5, 'LetsVenture': 5, 'Omidyar Network': 5, 'IDG Ventures': 5, 'Paytm': 5, 'Axilor Ventures': 5, 'Kae Capital': 5, 'SRI Capital': 5, 'India Educational Investment Fund': 5, 'India Quotient': 5, 'Undisclosed HNIs': 5, 'ah! Ventures': 5, 'Hyderabad Angels (at Startup Heroes event)': 5, 'Vijay Shekhar Sharma': 4, 'Sequoia India': 4, '

In [269]:
# subvertical
print("Unique:", df["SubVertical"].nunique())
print("Nulls:", df["SubVertical"].isna().sum())
print(df["SubVertical"].value_counts(dropna=False).to_dict())

Unique: 1942
Nulls: 936
{nan: 936, 'Online Lending Platform': 11, 'Online Pharmacy': 10, 'Food Delivery Platform': 8, 'Education': 5, 'Online Education Platform': 5, 'Online Lending': 5, 'Online Learning Platform': 5, 'Online lending platform': 5, 'Non-Banking Financial Company': 4, 'Online Food Delivery': 4, 'E-learning': 3, 'Online Food Delivery Platform': 3, 'Logistics': 3, 'Online Marketplace': 3, 'SaaS': 3, 'Agri-tech': 3, 'Online Insurance Platform': 3, 'Online Insurance Aggregator': 3, 'Online Furniture Store': 3, 'Online platform for Higher Education Services': 3, 'B2B Marketplace': 3, 'Online Gifting platform': 3, 'Online learning platform': 3, 'Online Payment Gateway': 3, 'ECommerce Marketplace': 3, 'Fitness Mobile App': 3, 'Data Analytics platform': 3, 'Agritech': 2, 'Mobile Wallet': 2, 'Financial Services': 2, 'Social Commerce': 2, 'Digital Lending Platform': 2, 'Wealth Management': 2, 'Cabs': 2, 'Brewery': 2, 'FinTech': 2, 'Optimization': 2, 'Artificial Intelligence': 2, '

In [270]:
# Leading/trailing whitespace
print((df["SubVertical"].fillna("")!=df["SubVertical"].fillna("").str.strip()).sum())
# Multiple spaces
print(  df["SubVertical"].str.contains(r"\s{2,}", regex=True, na=False).sum())

# Encoding issues
print(df["SubVertical"].str.contains(r"\\\\", regex=True, na=False).sum())

0
1
27


In [271]:
# removing encoding issues
df["SubVertical"] = (df["SubVertical"]
        .str.replace(r"\\\\xc2\\\\xa0", " ", regex=True)
        .str.replace(r"\\\\xe2\\\\x80\\\\x99", "'", regex=True)
        .str.replace(r"\\\\n", " ", regex=True)
)

# collapsing multiple spaces
df["SubVertical"] = df["SubVertical"].str.replace( r"\s+", " ", regex=True)

In [272]:
df["SubVertical"] = df["SubVertical"].str.strip()

In [273]:
#remarks column
print("Unique:", df["Remarks"].nunique())
print("Nulls:", df["Remarks"].isna().sum())
print(df["Remarks"].value_counts(dropna=False).to_dict())

Unique: 72
Nulls: 2625
{nan: 2625, 'Series A': 175, 'Series B': 63, 'Pre-Series A': 37, 'Series C': 28, 'Strategic Investment': 11, 'Series D': 11, 'Late Stage': 9, 'Strategic Funding': 6, 'pre-Series A': 6, 'At the 10 minute million event': 6, 'Bridge Round': 4, 'Bridge Funding': 2, '\\\\xc2\\\\xa0Series A': 2, 'Yet to Launch platform': 1, 'Series B (includes Debt financing)': 1, 'part of $40M Series B round': 1, 'Additional investment from parent company': 1, 'Additional Funding': 1, 'Yet to Launch': 1, 'Bridge funding': 1, 'Funding happened in Sept 2015': 1, 'Pre-Series A Bridge': 1, '2nd seed funding': 1, 'Series F ( More Details Here)': 1, 'Pre-Series A bridge round': 1, 'Pre-Series A Bridge round': 1, 'Strategic Funding (Series C)': 1, 'Strategic Investment (Majority Stake)': 1, 'Part of Series A raised in June 2015': 1, 'Part of $12M Series B funding': 1, 'thru Accelerator': 1, 'late Stage (part of $500M funding rnd)': 1, 'Late Stage (Alibaba @ 40% equity)': 1, 'Super angel roun

In [274]:
# Whitespace
print(( df["Remarks"].fillna("") !=df["Remarks"].fillna("").str.strip()).sum())

# Multiple spaces
print(df["Remarks"].str.contains(r"\s{2,}", regex=True, na=False).sum())

# Encoding
print(df["Remarks"].str.contains(r"\\\\", regex=True, na=False).sum())

0
0
5


In [275]:
remarks_replacements = {

    # Encoding
    "\\\\xc2\\\\xa0Series A": "Series A",
    "\\\\xc2\\\\xa0Series B": "Series B",
    "\\\\xc2\\\\xa0Late Stage": "Late Stage",

    # Pre-Series A
    "pre-Series A": "Pre-Series A",
    "pre Series-A": "Pre-Series A",
    "Pre Series-A": "Pre-Series A",
    "pre series A": "Pre-Series A",
    "pre-series A": "Pre-Series A",

    # Bridge
    "Bridge funding": "Bridge Funding",
    "Bridge round": "Bridge Round",

    # Funding
    "additional funding round": "Additional Funding"
}
df["Remarks"] = df["Remarks"].replace(investor_replacements)

In [276]:
#converting date to mysql friendly format
df["Date dd/mm/yyyy"] = df["Date dd/mm/yyyy"].dt.strftime("%Y-%m-%d")

In [277]:
print("shape: \n",df.shape)
print('info: \n',df.info())
print('describe: \n',df.describe())
print('dtypes: \n',df.dtypes)

shape: 
 (3044, 10)
<class 'pandas.DataFrame'>
RangeIndex: 3044 entries, 0 to 3043
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Sr No              3044 non-null   int64  
 1   Date dd/mm/yyyy    3044 non-null   str    
 2   Startup Name       3044 non-null   str    
 3   Industry Vertical  2873 non-null   str    
 4   SubVertical        2108 non-null   str    
 5   City  Location     2864 non-null   str    
 6   Investors Name     3020 non-null   str    
 7   InvestmentnType    3040 non-null   str    
 8   Amount in USD      2065 non-null   float64
 9   Remarks            419 non-null    str    
dtypes: float64(1), int64(1), str(8)
memory usage: 237.9 KB
info: 
 None
describe: 
              Sr No  Amount in USD
count  3044.000000   2.065000e+03
mean   1522.500000   1.842990e+07
std     878.871435   1.213734e+08
min       1.000000   1.600000e+04
25%     761.750000   4.700000e+05
50%    1522.500000 

In [278]:
df.head()

,Sr No,Date dd/mm/yyyy,Startup Name,Industry Vertical,SubVertical,City Location,Investors Name,InvestmentnType,Amount in USD,Remarks
0,1,2020-01-09,BYJU'S,EdTech,E-learning,Bengaluru,Tiger Global Management,Private Equity,200000000.0,NaN
1,2,2020-01-13,Shuttl,Transportation,App based shuttle service,Gurugram,Susquehanna Growth Equity,Series C,8048394.0,NaN
2,3,2020-01-09,Mamaearth,E-commerce,Retailer of baby and toddler products,Bengaluru,Sequoia Capital India,Series B,18358860.0,NaN
3,4,2020-01-02,https://www.wealthbucket.in/,FinTech,Online Investment,New Delhi,Vinod Khatumal,Pre-Series A,3000000.0,NaN
4,5,2020-01-02,Fashor,Fashion and Apparel,Embroiled Clothes For Women,Mumbai,Sprout Venture Partners,Seed Funding,1800000.0,NaN


In [279]:
df.rename(columns={"Sr No": "sr_no"}, inplace=True)
df.rename(columns={"Date dd/mm/yyyy": "funding_date"}, inplace=True)
df.rename(columns={"Startup Name": "startup_name"}, inplace=True)
df.rename(columns={"Industry Vertical": "industry_vertical"}, inplace=True)
df.rename(columns={"SubVertical": "sub_vertical"}, inplace=True)
df.rename(columns={"City  Location": "city_location"}, inplace=True)
df.rename(columns={"Investors Name": "investors_name"}, inplace=True)
df.rename(columns={"InvestmentnType": "investment_type"}, inplace=True)
df.rename(columns={"Amount in USD": "amount_usd"}, inplace=True)
df.rename(columns={"Remarks": "remarks"}, inplace=True)



import csv

df.to_csv("D:\Programming\Python\Data Analysis\Startup_Funding_Analysis\data\startup_clean.csv", index=False,na_rep="",quoting=csv.QUOTE_ALL)

<>:16: SyntaxWarning: "\P" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\P"? A raw string is also an option.
<>:16: SyntaxWarning: "\P" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\P"? A raw string is also an option.
C:\Users\dbari\AppData\Local\Temp\ipykernel_7228\2474615574.py:16: SyntaxWarning: "\P" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\P"? A raw string is also an option.
  df.to_csv("D:\Programming\Python\Data Analysis\Startup_Funding_Analysis\data\startup_clean.csv", index=False,na_rep="",quoting=csv.QUOTE_ALL)


In [280]:
#check the saved data
df_check=pd.read_csv(r'D:\Programming\Python\Data Analysis\Startup_Funding_Analysis\data\startup_clean.csv')

In [281]:
print(df_check.info())
print(df_check.head())

<class 'pandas.DataFrame'>
RangeIndex: 3044 entries, 0 to 3043
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   sr_no              3044 non-null   int64  
 1   funding_date       3044 non-null   str    
 2   startup_name       3044 non-null   str    
 3   industry_vertical  2873 non-null   str    
 4   sub_vertical       2108 non-null   str    
 5   city_location      2864 non-null   str    
 6   investors_name     3020 non-null   str    
 7   investment_type    3040 non-null   str    
 8   amount_usd         2065 non-null   float64
 9   remarks            419 non-null    str    
dtypes: float64(1), int64(1), str(8)
memory usage: 237.9 KB
None
   sr_no funding_date                  startup_name    industry_vertical  \
0      1   2020-01-09                        BYJU'S               EdTech   
1      2   2020-01-13                        Shuttl       Transportation   
2      3   2020-01-09              

In [282]:
print(df_check["amount_usd"].dtype)
print(df_check["amount_usd"].map(type).value_counts())

float64
amount_usd
<class 'float'>    3044
Name: count, dtype: int64


In [283]:
print(df_check["amount_usd"].map(type).value_counts())

amount_usd
<class 'float'>    3044
Name: count, dtype: int64


In [284]:
print(df_check["amount_usd"].describe())
print(df["amount_usd"].head(20))

count    2.065000e+03
mean     1.842990e+07
std      1.213734e+08
min      1.600000e+04
25%      4.700000e+05
50%      1.700000e+06
75%      8.000000e+06
max      3.900000e+09
Name: amount_usd, dtype: float64
0     200000000.0
1       8048394.0
2      18358860.0
3       3000000.0
4       1800000.0
5       9000000.0
6     150000000.0
7       6000000.0
8      70000000.0
9      50000000.0
10     20000000.0
11     12000000.0
12     30000000.0
13      5900000.0
14      2000000.0
15     50000000.0
16    231000000.0
17    150000000.0
18       486000.0
19      1500000.0
Name: amount_usd, dtype: float64
